In [ ]:
# %pip install implicit==0.7.2 requests==2.32.3 rectools[lightfm]==0.12.0 pandas==2.2.3 numpy==1.26.4 scipy==1.12.0 --quite

Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel $\rightarrow$ Restart) and then **run all cells** (in the menubar, select Cell $\rightarrow$ Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [1]:
NAME = "Юрпалов Сергей Николаевич"
COLLABORATORS = ""

---

# Домашнее задание №4. Матричная факторизация

## Задачи - 25 баллов (+5 доп баллов)
1. Ноутбук `mf.ipynb` - 20 баллов
- ImplicitALS - 4 балла
- SVD - 4 балла
- Dataset with features - 2 балла
- ImplicitALS with features - 5 баллов
- LightFM with features - 5 баллов
2. Имплементация модели в сервис - 5 баллов
- Пробить на Leaderboard порог `map@10 = 0.075`
- Если при этом используете MF (Implicit или LightFM) + ANN (nmslib, faiss, annoy и тд) - дополнительно 5 баллов
  
## Как сдать ноутбук `mf.ipynb` на проверку

1. Прогоните весь код ноутбука - проверьте, что нет ошибок и тесты проходят
2. Выложите готовый ноутбук в ваш репозиторий с сервисом из домашнего задания №1 по пути `notebooks/hw_4/mf.ipynb` в ветке `hw_4`
3. Проверьте, что есть доступ к вашему репозиторию для аккаунтов `https://github.com/feldlime`
4. Откройте PR в main ветку и добавьте в ревьюеры **своего ментора**
5. Не проводите мердж в `main` ветку, пока не увидите оценку за это ДЗ в ведомости. Файл с ноутбуком должен находиться в ветке `hw_4`

Обратите внимание, что сборка ноутбуков на проверку автоматизирована. В случае неправильного пути, имени файла или ветки (а также при отсутствии доступа у `@feldlime`) ваша работа не попадёт на проверку и получит `0` баллов.

Используемые библиотеки в рамках ДЗ
```bash
pip install implicit==0.7.2 requests==2.32.3 rectools[lightfm]==0.12.0 pandas==2.2.3 numpy==1.26.4 scipy==1.12.0
```

## Импорты и данные

In [2]:
import os
import os.path
import threadpoolctl

import requests

import warnings

from pathlib import Path

from itertools import product


import numpy as np
import pandas as pd

import zipfile as zf


from tqdm.auto import tqdm

from implicit.als import AlternatingLeastSquares

from lightfm import LightFM


from rectools import Columns

from rectools.metrics import MAP, MeanInvUserFreq

from rectools.dataset import Dataset

from rectools.models import PureSVDModel, ImplicitALSWrapperModel, LightFMWrapperModel, model_from_config


# For implicit ALS

os.environ["OPENBLAS_NUM_THREADS"] = "1"

threadpoolctl.threadpool_limits(1, "blas")

warnings.filterwarnings("ignore")

In [ ]:
data_path = os.environ.get("DATA_PATH")

Если вдруг у вас нет данных, то используйте закомментированный код

In [8]:
# %%time

# !wget -q https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip -O kion_train.zip
# !unzip -o kion_train.zip -x '__MACOSX/*'
# !rm kion_train.zip

Archive:  kion_train.zip
   creating: data_original/
  inflating: data_original/interactions.csv  
  inflating: data_original/users.csv  
  inflating: data_original/items.csv  
CPU times: user 24.4 ms, sys: 30.4 ms, total: 54.8 ms
Wall time: 3.96 s


In [ ]:
if data_path is None:
    data_path = "./data_original"

In [3]:
interactions = pd.read_csv(os.path.join(data_path, "interactions.csv"), parse_dates=["last_watch_dt"])
interactions = interactions.rename(columns={"total_dur": Columns.Weight, "last_watch_dt": Columns.Datetime})


users = pd.read_csv(os.path.join(data_path, "users.csv"))
items = pd.read_csv(os.path.join(data_path, "items.csv"))


print(interactions.shape)
interactions.head(5)

In [4]:
N_DAYS = 7

max_date = interactions["datetime"].max()
train = interactions[(interactions["datetime"] <= max_date - pd.Timedelta(days=N_DAYS))]
test = interactions[(interactions["datetime"] > max_date - pd.Timedelta(days=N_DAYS))]

catalog = train[Columns.Item].unique()

test_users = test[Columns.User].unique()
cold_users = set(test_users) - set(train[Columns.User])
test.drop(test[test[Columns.User].isin(cold_users)].index, inplace=True)
hot_users = test[Columns.User].unique()

In [5]:
K_RECOS = 10
map10 = MAP(k=K_RECOS)
RANDOM_SEED = 42
CPUS = os.cpu_count()

## ImplicitALS

### Ситуация:

Коллега вернулся из отпуска и вы вместе сели за улучшение модели. Внимательно изучив репозиторий библиотеки implicit вы увидели модель iALS и решаете попробовать ее в деле.

Чтобы работа была интереснее, вы заключаете пари с вашим коллегой о том, кто выбьет больше MAP@K на горячих пользователях. 

Правила пари: 
- Валидируемся на последней неделе (переменная `test`) и на горячих пользователях `hot_users`
- Можно собрать свой `Dataset` на основе `train`, трансформированного, если нужно
- Параметры модели задаются конфигом, которые будут передаваться в `model_from_config`

У вашего коллеги получилось выбить на ImplicitALS `MAP@K = 0.052`. Ваша задача побить его рекорд.

In [6]:
dataset = Dataset.construct(train)

In [7]:
config = {
    "cls": "ImplicitALSWrapperModel",
    "model": {
        "regularization": 0.1,
        "random_state": RANDOM_SEED,
        "num_threads": os.cpu_count(),
        "alpha": 0.01,
        "factors": 1,
        "iterations": 50,
    },
    "fit_features_together": False,
}

In [8]:
%%time
assert config['cls'] == 'ImplicitALSWrapperModel'

model = model_from_config(config)
model.fit(dataset)

recos = model.recommend(
    users=hot_users,
    dataset=dataset,
    k=K_RECOS,
    filter_viewed=True,
)
print(map10.calc(recos, test))

assert map10.calc(recos, test) >= 0.052

0.07624238298610563
CPU times: user 1min 31s, sys: 13min 16s, total: 14min 47s
Wall time: 57.7 s


## SVD

На ваш громкий спор с коллегой о том, что все дело в вашем удачном random seed, к вам подошел ваш лид. 

Узнав детали вашего спора, он дает вам комментарий, что iALS хороша, но погружение в матричную факторизацию следует начинать с `SVD`.

Вы переглянулись с коллегой и решаете уладить спор о random seed во втором раунде, используя новую модель.

Ваш коллега смогу выбить на SVD `MAP@K = 0.066`. Вы знаете, что делать.

In [9]:
def process_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    reference_date = df_copy[Columns.Datetime].max()
    df_copy["recency"] = (reference_date - df_copy[Columns.Datetime]).dt.days

    df_copy["log_total_dur"] = np.log1p(df_copy[Columns.Weight])

    df_copy["watched_pct_norm"] = df_copy["watched_pct"] / 100.0

    df_copy["watch_month"] = df_copy[Columns.Datetime].dt.month
    df_copy["watch_dayofweek"] = df_copy[Columns.Datetime].dt.dayofweek

    user_agg = (
        df_copy.groupby("user_id")
        .agg(
            total_dur_sum=(Columns.Weight, "sum"),
            total_dur_mean=(Columns.Weight, "mean"),
            recency_mean=("recency", "mean"),
            interactions_count=("user_id", "count"),
        )
        .reset_index()
    )

    item_agg = (
        df_copy.groupby("item_id")
        .agg(
            total_dur_sum=(Columns.Weight, "sum"),
            watched_pct_mean=("watched_pct", "mean"),
            interactions_count=("item_id", "count"),
        )
        .reset_index()
    )

    df_copy = df_copy.merge(user_agg, on="user_id", how="left", suffixes=("", "_user"))
    df_copy = df_copy.merge(item_agg, on="item_id", how="left", suffixes=("", "_item"))

    df_copy.drop(columns=["watched_pct", Columns.Weight], inplace=True)
    df_copy.rename(columns={"log_total_dur": Columns.Weight}, inplace=True)

    return df_copy

In [10]:
train2 = process_dataframe(train)

In [11]:
train2.head()

,user_id,item_id,datetime,recency,weight,watched_pct_norm,watch_month,watch_dayofweek,total_dur_sum,total_dur_mean,recency_mean,interactions_count,total_dur_sum_item,watched_pct_mean,interactions_count_item
0,176549,9506,2021-05-11,96,8.354910,0.72,5,1,342513,4176.987805,72.743902,82,25449476,59.973302,3858
1,699317,1659,2021-05-29,78,9.026177,1.00,5,5,390800,4652.380952,95.928571,84,5856997,70.303553,985
2,656683,7107,2021-05-09,98,2.397895,0.00,5,6,93,31.000000,92.333333,3,9677311,27.931691,16279
3,864613,7638,2021-07-05,41,9.580800,1.00,7,0,66905,7433.888889,37.888889,9,8986446,49.132697,1153
4,964868,9506,2021-04-30,107,8.813736,1.00,4,4,10493,3497.666667,99.666667,3,25449476,59.973302,3858


In [12]:
dataset2 = Dataset.construct(train2)

In [13]:
config = {
    "cls": "PureSVDModel",
    "factors": 1,
    "maxiter": 100,
    "random_state": RANDOM_SEED,
}

In [14]:
%%time
assert config['cls'] == 'PureSVDModel'

model = model_from_config(config)
model.fit(dataset2)

recos = model.recommend(
    users=hot_users,
    dataset=dataset2,
    k=K_RECOS,
    filter_viewed=True,
)
print(map10.calc(recos, test))

assert map10.calc(recos, test) >= 0.066

0.07822653775342518
CPU times: user 9.69 s, sys: 4.99 s, total: 14.7 s
Wall time: 11 s


## Dataset with features

"Ну это ни в какие ворота!" - восклицает ваш коллега, увидев ваш победный конфиг. Из другого угла опенспейса доносится "А я говорил" от вашего лида.

В это время к вам сзади подходит продакт и интересуется предметом вашего спора.

Рассказав про особенности найденных вами моделей, он просит вас в них докинуть фичи, ведь на одних взаимодействиях далеко не уедешь.

Вы согласились, ведь это отличная возможность продолжить пари. Соберите `Dataset` с фичами по пользователям и итемам, который вы будете использовать дальше


**User**

In [15]:
users.fillna("Unknown", inplace=True)
users = users.loc[users[Columns.User].isin(train[Columns.User])].copy()

In [16]:
user_features_frames = []
for feature in tqdm(["sex", "age", "income"]):
    feature_frame = users.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)

user_features = pd.concat(user_features_frames)
user_features.head()

  0%|          | 0/3 [00:00<?, ?it/s]

,id,value,feature
0,973171,М,sex
1,962099,М,sex
3,721985,Ж,sex
4,704055,Ж,sex
5,1037719,М,sex


In [17]:
user_features.query(f"id == 973171")

,id,value,feature
0,973171,М,sex
0,973171,age_25_34,age
0,973171,income_60_90,income


**Item**

In [18]:
items = items.loc[items[Columns.Item].isin(train[Columns.Item])].copy()

In [19]:
items["genre"] = items["genres"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
genre_feature = items[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature.head()

,id,value,feature
0,10711,драмы,genre
0,10711,зарубежные,genre
0,10711,детективы,genre
0,10711,мелодрамы,genre
1,2508,зарубежные,genre


In [20]:
content_feature = items.reindex(columns=[Columns.Item, "content_type"])
content_feature.columns = ["id", "value"]
content_feature["feature"] = "content_type"

In [22]:
item_features = pd.concat((genre_feature, content_feature))

In [23]:
item_features.query(f"id == 1000")

,id,value,feature
6162,1000,боевики,genre
6162,1000,мелодрамы,genre
6162,1000,комедии,genre
6162,1000,film,content_type


In [24]:
dataset_with_features = Dataset.construct(
    interactions_df=train,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type"],
)

In [25]:
assert (dataset_with_features.user_features is not None) and (dataset_with_features.item_features is not None)

## ImplicitALS with features

Собрав датасет с фичами вы готовы к третьему раунду пари. 

Вы решаете начать снова с `iALS`, до сих пор удивляясь результатам модели `SVD`.

Ваш коллега изучил вашу технику подбора random seed и хитро улыбается вам.

Он смог выбить `MAP@K = 0.073`, теперь ваш ход.

In [28]:
config = {
    "cls": "ImplicitALSWrapperModel",
    "model": {
        "regularization": 0.1,
        "random_state": RANDOM_SEED,
        "num_threads": os.cpu_count(),
        "alpha": 0.01,
        "factors": 1,
        "iterations": 50,
    },
    "fit_features_together": True,
}

In [29]:
%%time
assert config['cls'] == 'ImplicitALSWrapperModel'

model = model_from_config(config)
model.fit(dataset_with_features)

recos = model.recommend(
    users=hot_users,
    dataset=dataset_with_features,
    k=K_RECOS,
    filter_viewed=True,
)
print(map10.calc(recos, test))

assert map10.calc(recos, test) >= 0.073

0.07708523543626086
CPU times: user 8min 51s, sys: 25min 6s, total: 33min 58s
Wall time: 2min 17s


## LightFM with features

И снова ор выше гор, ваш пайплайн подготовки датасета помог вам в очередной раз обойти вашего коллегу.

Не зная, к чему еще аппелировать, он зовет вашего старшего коллегу, чтобы тот внимательно изучил полученные результаты.

"iALS с фичами это хорошо, но тут стоит попробовать факторизационные машины, попробуйте `LightFM`" - заключает он. Вы переключаетесь на изучение новой библиотеки, предвкушая финальный раунд.

Ваш коллега смог выжать из своего обновленного `Dataset` и `LightFM` скор `MAP@10 = 0.08`. Последний рывок.

In [30]:
config = {
    "cls": "LightFMWrapperModel",
    "model": {
        "loss": "warp",
        "random_state": RANDOM_SEED,
    },
}

In [31]:
%%time
assert config['cls'] == 'LightFMWrapperModel'

model = model_from_config(config)
model.fit(dataset_with_features)

recos = model.recommend(
    users=hot_users,
    dataset=dataset_with_features,
    k=K_RECOS,
    filter_viewed=True,
)
print(map10.calc(recos, test))

assert map10.calc(recos, test) >= 0.08

0.08090227236094076
CPU times: user 12 s, sys: 76.2 ms, total: 12 s
Wall time: 12 s


## Сервис

Эта битва была легендарной, но вот за спиной опять появляется ваш продакт. 

"Время катить АБ" - говорит он в пятницу вечером. Делать нечего, вы собираете ваши наработки и определяете совместный фронт работ.

В прод должна заехать лучшая модель, которая побьет текущей модели в проде в `MAP@10 = 0.075`.

Также есть бонус от вашего старшего коллеги, который вам советует присмотреться к `Approximate nearest neighbours`, например к `nmslib`.

"Если сможешь обернуть в ANN, то на следующем годовом ревью получишь от меня оценку отлично" - сказал он. Изучить новые технологии и получить повышение - идеально, заключаете вы и бросаетесь в бой.

# Вместо заключения

## Задачи - 25 баллов (30 баллов с доп задачей по ANN)
1. Ноутбук `mf.ipynb` - 20 баллов
- SVD - 5 баллов
- ImplicitALS - 5 баллов
- ImplicitALS with features - 5 баллов
- LightFM with features - 5 баллов
2. Имплементация модели в сервис - 5 баллов
- Пробить на Leaderboard порог `map@10 = 0.075`
- Если при этом используете MF (Implicit или LightFM) + ANN (nmslib, faiss, annoy и тд) - дополнительно 5 баллов
  
## Как сдать ноутбук `mf.ipynb` на проверку

1. Прогоните весь код ноутбука - проверьте, что нет ошибок и тесты проходят
2. Выложите готовый ноутбук в ваш репозиторий с сервисом из домашнего задания №1 по пути `notebooks/hw_4/mf.ipynb` в ветке `hw_4`
3. Проверьте, что есть доступ к вашему репозиторию для аккаунтов `https://github.com/feldlime`
4. Откройте PR в main ветку и добавьте в ревьюеры **своего ментора**
5. Не проводите мердж в `main` ветку, пока не увидите оценку за это ДЗ в ведомости. Файл с ноутбуком должен находиться в ветке `hw_4`

Обратите внимание, что сборка ноутбуков на проверку автоматизирована. В случае неправильного пути, имени файла или ветки (а также при отсутствии доступа у `@feldlime`) ваша работа не попадёт на проверку и получит `0` баллов.